# Hi-EF Phase 1: post-audit diagnostics

This notebook runs the frozen Phase-1 calibration, residual, and same-checkpoint branch diagnostics. Attach the saved canonical matrix output and `ptrnghieu/hi-ef-features-v2`, enable a T4 GPU and Internet, then choose **Save Version → Save & Run All**. It does not train, select, or test a model.

In [ ]:
from pathlib import Path
import json
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
FEATURES = Path('/kaggle/input/datasets/ptrnghieu/hi-ef-features-v2')
OUTPUT = Path('/kaggle/working/canonical_phase1_diagnostics')

if not (REPO / '.git').exists():
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
else:
    subprocess.run([
        'git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'
    ], check=True)

assert (FEATURES / '01_00059.pt').is_file(), 'Feature dataset is not attached'
candidates = []
for path in Path('/kaggle/input').rglob('canonical_residual_matrix_summary.json'):
    try:
        payload = json.loads(path.read_text())
    except Exception:
        continue
    if payload.get('protocol') == 'canonical-contextual-affective-residual-validation-matrix-v1':
        candidates.append(path)
assert len(candidates) == 1, f'Expected exactly one canonical matrix output, found: {candidates}'
MATRIX = candidates[0].parent
MANIFEST = REPO / 'experiments/manifests/source_folder_split_seed42.csv'
commit = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True
).strip()
print('Ready at commit:', commit)
print('Matrix input:', MATRIX)

In [ ]:
import os
environment = {**os.environ, 'PYTHONPATH': str(REPO / 'experiments')}
subprocess.run([
    'python', '-m', 'unittest',
    str(REPO / 'experiments/test_canonical_diagnostic_metrics.py'),
    str(REPO / 'experiments/test_contextual_affective_residual.py'),
], cwd=REPO, env=environment, check=True)

subprocess.run([
    'python', str(REPO / 'experiments/run_canonical_phase1_diagnostics.py'),
    '--matrix-dir', str(MATRIX),
    '--manifest', str(MANIFEST),
    '--features-dir', str(FEATURES),
    '--output-dir', str(OUTPUT),
    '--bootstrap-replicates', '5000',
], cwd=REPO, env=environment, check=True)

In [ ]:
import pandas as pd

summary_path = OUTPUT / 'canonical_phase1_diagnostic_summary.json'
summary = json.loads(summary_path.read_text())
assert summary['diagnostic_only'] is True
assert summary['test_evaluated'] is False
assert summary['partitions_touched'] == ['validation']
display(pd.DataFrame(summary['calibration_aggregates']))
print(json.dumps(summary['residual_aggregates'], indent=2))
print(json.dumps(summary['branch_intervention_hierarchical_bootstrap'], indent=2))
print('Download:', summary_path)
for filename in summary['output_files']:
    print('Download:', OUTPUT / filename)